In [ ]:
#@title Imports
# ============================================
# Cell 1
# Install + import dependencies
# ============================================

# Install OpenCV if needed
!pip install opencv-python-headless -q

import cv2
import numpy as np
import matplotlib.pyplot as plt
import json
import base64
from google.colab import files
from google.colab import output
from IPython.display import display, HTML

print("Libraries loaded successfully")

In [ ]:
#@title Load Vid
# ============================================
# Cell 2
# Load video + extract first frame
# ============================================

# -----------------------------
# Hardcoded video path
# -----------------------------
# Edit this to point at your video file. If you're uploading
# manually via the Colab file browser (left sidebar > Files),
# the file will land under /content/.
VIDEO_PATH = "/content/C1_20260804_133401.mp4"

print("Loaded video:")
print(VIDEO_PATH)


# -----------------------------
# Read first frame
# -----------------------------

cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    raise IOError("Could not open video")


ret, frame = cap.read()

cap.release()


if not ret:
    raise IOError("Could not read first frame")


# Convert BGR -> RGB for matplotlib
frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)


# Store dimensions
HEIGHT, WIDTH = frame_rgb.shape[:2]

print()
print("Frame dimensions:")
print(f"Width:  {WIDTH}")
print(f"Height: {HEIGHT}")


# Display first frame
plt.figure(figsize=(10,8))
plt.imshow(frame_rgb)
plt.axis("off")
plt.title("First frame of video")
plt.show()

In [ ]:
#@title Select Points
# ============================================
# Cell 3
# Interactive ROI selection (HTML5 canvas + JS)
#
# NOTE: The original Plotly FigureWidget "on_click"
# approach does not reliably fire in Colab's widget
# manager, which is why clicked_points stayed empty.
# This cell instead draws the frame on an HTML canvas
# and sends click coordinates back to Python through
# google.colab.output.register_callback, which is the
# supported way to do this in Colab.
# ============================================

clicked_points = []

# -----------------------------
# Circle-from-3-points helper
# -----------------------------
def calculate_circle(points):

    (x1, y1), (x2, y2), (x3, y3) = points

    temp = x2**2 + y2**2

    bc = (x1**2 + y1**2 - temp) / 2

    cd = (temp - x3**2 - y3**2) / 2

    det = (
        (x1 - x2) * (y2 - y3)
        -
        (x2 - x3) * (y1 - y2)
    )

    if abs(det) < 1e-10:
        raise ValueError("Points are collinear")

    cx = (
        bc * (y2 - y3)
        -
        cd * (y1 - y2)
    ) / det

    cy = (
        (x1 - x2) * cd
        -
        (x2 - x3) * bc
    ) / det

    r = np.sqrt(
        (cx - x1) ** 2 +
        (cy - y1) ** 2
    )

    return int(cx), int(cy), int(r)


# -----------------------------
# Callbacks invoked from JS
# -----------------------------
def register_point(x, y):
    global clicked_points

    clicked_points.append((float(x), float(y)))
    result = {"count": len(clicked_points)}

    if len(clicked_points) == 3:
        cx, cy, r = calculate_circle(clicked_points)
        result.update({"done": True, "cx": cx, "cy": cy, "r": r})

        print("\nROI CALCULATED")
        print("----------------")
        print(f"ROI_CX = {cx}")
        print(f"ROI_CY = {cy}")
        print(f"ROI_R  = {r}")
    else:
        result["done"] = False

    return result


def reset_points():
    global clicked_points
    clicked_points = []
    print("Points reset. Click 3 points again.")
    return {"status": "reset"}


output.register_callback("notebook.registerPoint", register_point)
output.register_callback("notebook.resetPoints", reset_points)


# -----------------------------
# Encode frame + build display canvas
# -----------------------------

# Scale the display so very large frames still fit on screen.
# Coordinates are scaled back up to full resolution in JS before
# being sent to Python, so ROI values stay in original pixel space.
MAX_DISPLAY_WIDTH = 900
scale = min(1.0, MAX_DISPLAY_WIDTH / WIDTH)
DISPLAY_WIDTH = int(WIDTH * scale)
DISPLAY_HEIGHT = int(HEIGHT * scale)

_, buffer = cv2.imencode(".png", cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2BGR))
img_b64 = base64.b64encode(buffer).decode("utf-8")

html_code = f"""
<div>
  <canvas id="roiCanvas" width="{DISPLAY_WIDTH}" height="{DISPLAY_HEIGHT}"
          style="border:1px solid #444; cursor:crosshair;"></canvas>
  <div style="margin-top:8px; font-family:sans-serif;">
    <button id="resetBtn">Reset points</button>
    <span id="status" style="margin-left:10px;">Click 3 points around the circle edge</span>
  </div>
</div>
<script>
(function() {{
  const canvas = document.getElementById('roiCanvas');
  const ctx = canvas.getContext('2d');
  const img = new Image();
  const scaleX = {WIDTH} / {DISPLAY_WIDTH};
  const scaleY = {HEIGHT} / {DISPLAY_HEIGHT};
  let points = [];

  img.onload = function() {{
    ctx.drawImage(img, 0, 0, {DISPLAY_WIDTH}, {DISPLAY_HEIGHT});
  }};
  img.src = "data:image/png;base64,{img_b64}";

  function redraw() {{
    ctx.clearRect(0, 0, canvas.width, canvas.height);
    ctx.drawImage(img, 0, 0, {DISPLAY_WIDTH}, {DISPLAY_HEIGHT});
    ctx.fillStyle = 'red';
    points.forEach(p => {{
      ctx.beginPath();
      ctx.arc(p.dx, p.dy, 5, 0, 2 * Math.PI);
      ctx.fill();
    }});
  }}

  canvas.addEventListener('click', async function(evt) {{
    if (points.length >= 3) return;  // ignore extra clicks until reset

    const rect = canvas.getBoundingClientRect();
    const dx = evt.clientX - rect.left;
    const dy = evt.clientY - rect.top;

    points.push({{dx: dx, dy: dy}});
    redraw();

    const fullX = dx * scaleX;
    const fullY = dy * scaleY;

    const result = await google.colab.kernel.invokeFunction(
      'notebook.registerPoint', [fullX, fullY], {{}}
    );
    const data = result.data['application/json'];

    if (data.done) {{
      document.getElementById('status').innerText =
        'ROI calculated! center=(' + data.cx + ', ' + data.cy + ') r=' + data.r;
    }} else {{
      document.getElementById('status').innerText =
        'Points selected: ' + data.count + ' / 3';
    }}
  }});

  document.getElementById('resetBtn').addEventListener('click', async function() {{
    points = [];
    redraw();
    await google.colab.kernel.invokeFunction('notebook.resetPoints', [], {{}});
    document.getElementById('status').innerText = 'Click 3 points around the circle edge';
  }});
}})();
</script>
"""

display(HTML(html_code))

In [ ]:
#@title Save
# ============================================
# Cell 4
# Export ROI calibration
# ============================================

# Verify that ROI was created
if len(clicked_points) != 3:
    raise RuntimeError(
        "ROI has not been calculated. "
        "Run Cell 3 and click three points on the canvas."
    )


# Calculate final ROI values
ROI_CX, ROI_CY, ROI_R = calculate_circle(clicked_points)


# -----------------------------
# Display output
# -----------------------------

print("================================")
print("ROI calibration complete")
print("================================")

print()
print(f"ROI_CX = {ROI_CX}")
print(f"ROI_CY = {ROI_CY}")
print(f"ROI_R  = {ROI_R}")


# -----------------------------
# Save JSON configuration
# -----------------------------

roi_config = {
    "roi_cx": int(ROI_CX),
    "roi_cy": int(ROI_CY),
    "roi_r": int(ROI_R)
}


ROI_FILE = "roi_config.json"


with open(ROI_FILE, "w") as f:
    json.dump(
        roi_config,
        f,
        indent=4
    )


print()
print("Saved:")
print(ROI_FILE)


# -----------------------------
# Print tracker snippet
# -----------------------------

print("\nCopy into tracking script:\n")

print(
f"""
ROI_CX = {ROI_CX}
ROI_CY = {ROI_CY}
ROI_R  = {ROI_R}
"""
)

In [ ]:
#@title Check ROI
# ============================================
# Cell 5
# Verify ROI selection
# ============================================

# -----------------------------
# Load saved ROI
# -----------------------------

with open("roi_config.json", "r") as f:
    roi_config = json.load(f)


ROI_CX = roi_config["roi_cx"]
ROI_CY = roi_config["roi_cy"]
ROI_R  = roi_config["roi_r"]


print("Loaded ROI:")
print(f"Center: ({ROI_CX}, {ROI_CY})")
print(f"Radius: {ROI_R}px")


# -----------------------------
# Create ROI mask
# -----------------------------

mask = np.zeros(
    (HEIGHT, WIDTH),
    dtype=np.uint8
)


cv2.circle(
    mask,
    (ROI_CX, ROI_CY),
    ROI_R,
    255,
    -1
)


# Apply mask
roi_frame = frame_rgb.copy()

roi_frame[mask == 0] = 0


# -----------------------------
# Draw ROI outline
# -----------------------------

overlay = roi_frame.copy()


cv2.circle(
    overlay,
    (ROI_CX, ROI_CY),
    ROI_R,
    (255, 0, 0),
    3
)


# -----------------------------
# Display
# -----------------------------

plt.figure(figsize=(10,8))

plt.imshow(overlay)

plt.scatter(
    ROI_CX,
    ROI_CY,
    s=50
)

plt.title(
    "Final ROI used for tracking"
)

plt.axis("off")

plt.show()